In [1]:
import pandas as pd
import talib
import pandas_ta as ta
initial_df = pd.read_csv("btcusdt_1h.csv")

final_df = initial_df.copy()
initial_df

,datetime,open,high,low,close,volume
0,2018-01-01 05:30:00,13715.65,13715.65,13400.01,13529.01,443.356199
1,2018-01-01 06:30:00,13528.99,13595.89,13155.38,13203.06,383.697006
2,2018-01-01 07:30:00,13203.00,13418.43,13200.00,13330.18,429.064572
3,2018-01-01 08:30:00,13330.26,13611.27,13290.00,13410.03,420.087030
4,2018-01-01 09:30:00,13434.98,13623.29,13322.15,13601.01,340.807329
...,...,...,...,...,...,...
35203,2022-01-12 01:30:00,42972.04,43095.26,42692.19,42800.38,1219.601780
35204,2022-01-12 02:30:00,42797.62,42823.69,42643.74,42659.20,702.103800
35205,2022-01-12 03:30:00,42664.71,42776.14,42597.41,42713.13,561.859930
35206,2022-01-12 04:30:00,42713.12,42886.28,42633.97,42729.29,681.142010


In [2]:
atr = talib.ATR(final_df['high'], final_df['low'], final_df['close'], timeperiod=15)

# Assign ATR values to a new column in your DataFrame
final_df['atr'] = atr

In [58]:
from ta.volatility import BollingerBands

# Initialize Bollinger Bands Indicator
indicator_bb = BollingerBands(close=final_df["close"], window=25, window_dev=2.5)
curr_sig=0
closed=True
final_df['bb_bbm_vals'] = indicator_bb.bollinger_mavg()
final_df['bb_bbh_vals'] = indicator_bb.bollinger_hband()
final_df['bb_bbl_vals'] = indicator_bb.bollinger_lband()

In [ ]:
from ta.utils import dropna
from ta.volume import OnBalanceVolumeIndicator

# Initialize On Balance Volume Indicator
indicator_obv = OnBalanceVolumeIndicator(close=final_df["close"],volume=final_df["volume"])
final_df['obv_values'] = indicator_obv.on_balance_volume()
#ema = exponential moving average
final_df["obv_ema"] = final_df['obv_values'].ewm(span=200, adjust=False).mean()

In [71]:
final_df["ema_25"]=final_df["close"].ewm(span=25, adjust=False).mean()
final_df["ema_50"]=final_df["close"].ewm(span=40, adjust=False).mean()
final_df["ema_70"]=final_df["close"].ewm(span=60, adjust=False).mean()

#condition to enter long trade
def strat_long_entry(final_df,bar):

    if final_df["ema_25"].iloc[bar]>final_df["ema_50"].iloc[bar] :
        return True
    else:
        return False

# condition to enter short trade
def strat_short_entry(final_df,bar):
    if final_df["ema_25"].iloc[bar]<final_df["ema_50"].iloc[bar] :
        return True
    else:
        return False

#condition to exit long trade
def strat_long_exit(final_df,bar):

    if final_df["ema_25"].iloc[bar]<final_df["ema_70"].iloc[bar] and final_df["ema_25"].iloc[bar-1]>=final_df["ema_70"].iloc[bar-1]:
        return True
    else:
        return False

#condition to exit short trade
def strat_short_exit(final_df,bar):
    if final_df["ema_25"].iloc[bar]>final_df["ema_70"].iloc[bar] and final_df["ema_25"].iloc[bar-1]<=final_df["ema_70"].iloc[bar-1]:
        return True
    else:
        return False

#stop loss conditions for long and short trades
def long_stop_loss(bar,long_entry_price):
    if long_entry_price==None:
        return False
    elif final_df["close"].iloc[bar]<long_entry_price-2.6*final_df["atr"].iloc[bar]\
        and final_df["bb_bbl_vals"].iloc[bar]>final_df["close"].iloc[bar]:
        return True
    else:
        return False
    

def short_stop_loss(bar,short_entry_price):
    
    if short_entry_price==None:
        return False
    elif final_df["close"].iloc[bar]>short_entry_price+2.6*final_df["atr"].iloc[bar]\
    and final_df["bb_bbh_vals"].iloc[bar]<final_df["close"].iloc[bar]:
        return True
    else:
        return False

In [72]:

signals_final=[0,0]
temp=[]
curr_sig=0
closed=True
import statistics
from statistics import mode
    #### set conditions for long_entry and long_exit
print(len(final_df))
long_entry_price=None
short_entry_price=None
for i in range(2,len(final_df)):
    if final_df.iloc[i].isna().any():
        signals_final.append(0)

    else:
        dic={}
        long_entry=strat_long_entry(final_df,i)

        short_exit=strat_short_exit(final_df,i)
        short_entry=strat_short_entry(final_df,i)
        long_exit=strat_long_exit(final_df,i)
        #stop loss ocnditions
        con1=short_stop_loss(i,short_entry_price)
        con2=long_stop_loss(i,long_entry_price)


        if long_entry or short_exit or con1:
              #go long
              if long_entry and closed==True:
                  #previous trade closed, can open new one
                  signals_final.append(1)
                  long_entry_price=final_df["close"].iloc[i]
                  closed=False
                  curr_sig=1
              else:
                                       
                  if ((con1 or short_exit)and curr_sig==-1):
                      closed=True
                      curr_sig=0
                      signals_final.append(1)
                      short_entry_price=None
                  else:
                      signals_final.append(0)
          # set -1
        elif short_entry or long_exit or con2:
              if short_entry and closed==True:
                  signals_final.append(-1)
                  short_entry_price=final_df["close"].iloc[i]
                  closed=False
                  curr_sig=-1
              else:
                  
                  if (long_exit or con2 ) and curr_sig==1:
                      
                      closed=True
                      curr_sig=0
                      signals_final.append(-1)
                      long_entry_price=None
                  else:
                      signals_final.append(0)
        else:
            signals_final.append(0)


35208


In [73]:
final_signals_values=pd.DataFrame(signals_final)
initial_df["signals"]=signals_final

initial_df.to_csv("output_new.csv")